### Read the Bronze Table

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_orders=spark.table("workspace.ecommerce_project.bronze_orders")
incremental_bronze_orders = (
    bronze_orders.filter(
        F.col("batch_id") == 'BATCH002'
    )
)
incremental_bronze_orders.show(incremental_bronze_orders.count())

+---------+-----------+------------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|     customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+------------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------

### Verify the Data

In [0]:
bronze_record_count=incremental_bronze_orders.count()
print(f"Bronze record count: {bronze_record_count}")

Bronze record count: 45


In [0]:
incremental_bronze_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- order_amount: double (nullable = true)
 |-- final_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- delivery_date: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_state: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- ingestion_timestamp: timestamp

### Understand the Columns

In [0]:
incremental_bronze_orders.columns

['order_id',
 'customer_id',
 'customer_name',
 'customer_email',
 'customer_phone',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_percent',
 'order_amount',
 'final_amount',
 'payment_method',
 'payment_status',
 'order_status',
 'order_date',
 'ship_date',
 'delivery_date',
 'shipping_city',
 'shipping_state',
 'pincode',
 'ingestion_timestamp',
 'source_file',
 'batch_id',
 'source_system',
 'pipeline_id']

### Initial Data Profiling

In [0]:


null_summary_df=incremental_bronze_orders.select(
    *[F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in bronze_orders.columns
    ]
)
null_summary_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

In [0]:
text_columns = [
    "customer_name",
    "customer_email",
    "customer_phone",
    "product_name",
    "category",
    "payment_method",
    "payment_status",
    "order_status",
    "shipping_city",
    "shipping_state",
    "pincode"
]

In [0]:
blank_summary_df=incremental_bronze_orders.select(
    *[
        F.sum((
        F.col(column).isNotNull() &
        F.trim((F.col(column))== "")
        ).cast("int")
        ).alias(column)
        for column in text_columns
    ]
)
blank_summary_df.show()

+-------------+--------------+--------------+------------+--------+--------------+--------------+------------+-------------+--------------+-------+
|customer_name|customer_email|customer_phone|product_name|category|payment_method|payment_status|order_status|shipping_city|shipping_state|pincode|
+-------------+--------------+--------------+------------+--------+--------------+--------------+------------+-------------+--------------+-------+
|            0|             0|             0|           0|       0|             0|             0|           0|            0|             0|      0|
+-------------+--------------+--------------+------------+--------+--------------+--------------+------------+-------------+--------------+-------+



### Record Our Starting Point

In [0]:
bronze_record_count=incremental_bronze_orders.count()
print(f"Starting Bronze records: {bronze_record_count}")

Starting Bronze records: 45


### Investigate customer_name

In [0]:
display(
    incremental_bronze_orders
    .groupBy("customer_name")
    .count()
    .orderBy(F.desc("count"))
)

customer_name,count
Manisha Nair,2
Priya Siddiqui,2
Rahul Subramaniam,2
Karthik Murthy,2
Radha Pillai,2
Selvam Varma,2
Vinod Chowdary,2
Mohammed Achari,2
Manoj Kumar,2
,1


In [0]:
customer_name_issues_df=(
    incremental_bronze_orders.filter(
    F.col("customer_name").isNull() |
    (F.trim(F.col("customer_name")) == "")
    )
)
customer_name_issues_df.show()

+---------+-----------+-------------+--------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|customer_name|      customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-------------+--------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+--

### Investigate customer_email

In [0]:
customer_email_issues_df=(
    incremental_bronze_orders.filter(
        F.col("customer_email").isNull() |
        F.trim((F.col("customer_email")) == "")
    )
)
customer_email_issues_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

### Check Email Format

In [0]:
email_pattern=r"^[A-Za-z.%+-_]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_email_df=incremental_bronze_orders.filter(
    F.col("customer_email").isNotNull() &
    F.trim((F.col("customer_email"))!="") &
    ~F.col("customer_email").rlike(email_pattern)
)
invalid_email_df.select(
    "order_id",
    "customer_name",
    "customer_email"
).show(truncate=False)

+--------+-------------+--------------+
|order_id|customer_name|customer_email|
+--------+-------------+--------------+
+--------+-------------+--------------+



### Normalize the Name First

In [0]:
order_name_normalized_df=incremental_bronze_orders.withColumn(
    "customer_name",
    F.when(
        F.trim(F.col("customer_name")) == "",None
    )
    .otherwise(
        F.trim(F.col("customer_name"))
    )
)
order_name_normalized_df.show(truncate=False)

+---------+-----------+---------------+-----------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+
|order_id |customer_id|customer_name  |customer_email               |customer_phone|product_id|product_name       |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+---------+-----------+---------------+-----------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+----

### Normalize the Email

In [0]:
orders_name_email_normalized_df=order_name_normalized_df.withColumn(
    "customer_email",
    F.when(
        F.trim(F.col("customer_email")) == "",None
    ).otherwise(
        F.lower(F.trim(F.col("customer_email")))
    )
)
orders_name_email_normalized_df.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--------

### Derive Missing Customer Names

In [0]:
orders_name_email_normalized_df.filter(F.col("customer_name").isNull()).show(truncate=False)

+---------+-----------+-------------+-------------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+
|order_id |customer_id|customer_name|customer_email           |customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+---------+-----------+-------------+-------------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+------------

In [0]:
order_name_email_recovered_df=(
    orders_name_email_normalized_df.withColumn(
        "customer_name",
        F.when(
            F.col("customer_name").isNull() &
            F.col("customer_email").rlike(email_pattern),
            F.initcap(
                F.regexp_replace(
                    F.split(F.col("customer_email"),"@")[0],
                    r"\.",
                    " "
                )
            )
        ).otherwise(F.col("customer_name"))
        )
)
order_name_email_recovered_df.filter(F.col("order_id")=="ORD100896").show(truncate=False)

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

### Verify the Recovery

In [0]:
display(
    order_name_email_recovered_df.filter(
        F.col("customer_name").isNotNull()
    ).select(
        "order_id",
        "customer_name",
        "customer_email"
    )
)

order_id,customer_name,customer_email
ORD100208,Rahul Pillai,rahul.pillai@outlook.com
ORD100244,Vinod Chowdary,vinod.chowdary@hotmail.com
ORD100779,Arun Balan,arun.balan@outlook.com
ORD100778,Zara Sundaram,zara.sundaram@yahoo.com
ORD100428,Kalpana Rajan,kalpana.rajan@outlook.com
ORD100923,Meena Kumar,meena.kumar@rediffmail.com
ORD100111,Rakesh Murthy,rakesh.murthy@outlook.com
ORD100958,Selvam Varma,selvam.varma@yahoo.com
ORD100959,Ashwin Menon,ashwin.menon@gmail.com
ORD100960,Priya Siddiqui,priya.siddiqui@yahoo.com


### Identify Records That Cannot Be Recovered

In [0]:
unrecoverable_customer_df=order_name_email_recovered_df.filter(
    F.col("customer_name").isNull() &
    (
        F.col("customer_email").isNull() |
        ~F.col("customer_email").rlike(email_pattern)
    )
)
display(unrecoverable_customer_df.select(
    "order_id",
    "customer_id",
    "customer_name",
    "customer_email"
))

order_id,customer_id,customer_name,customer_email


In [0]:
invalid_email_df.show(truncate=False)

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

### Investigate customer_phone

In [0]:
incremental_bronze_orders.filter(
    F.col("customer_phone").isNull() |
    (F.trim(F.col("customer_phone")) == "")
).select(
    "order_id",
    "customer_id",
    "customer_phone"
).show(truncate=False)

+--------+-----------+--------------+
|order_id|customer_id|customer_phone|
+--------+-----------+--------------+
+--------+-----------+--------------+



In [0]:
normalized_phone_df=incremental_bronze_orders.withColumn(
    "customer_phone",
    F.regexp_replace(
        F.col("customer_phone"),
        r"^\+91[-\s]*",
        ""
        )
)

In [0]:
normalized_phone_df=normalized_phone_df.withColumn(
    "customer_phone",
    F.regexp_replace(
        F.col("customer_phone"),
        r"[^0-9]",
        ""
    )
)

In [0]:
normalized_phone_df.select(
    "order_id",
    "customer_phone"
).show()

+---------+--------------+
| order_id|customer_phone|
+---------+--------------+
|ORD100208|    8357109965|
|ORD100244|    6248578017|
|ORD100779|    9406284028|
|ORD100778|    8249926919|
|ORD100428|    9655742829|
|ORD100923|    6178663096|
|ORD100111|    7750572325|
|ORD100958|    9952266893|
|ORD100959|    6200140141|
|ORD100960|    6359154084|
|ORD100011|    9334978881|
|ORD100365|    9334978881|
|ORD100954|    8481469012|
|ORD100953|    8816070802|
|ORD100382|    8141078352|
|ORD100099|    9719106699|
|ORD100077|    7960514523|
|ORD100298|    9952266893|
|ORD100963|    6807577342|
|ORD100313|    8327268503|
+---------+--------------+
only showing top 20 rows


In [0]:
normalized_phone_df.select(
    "customer_phone"
).distinct().show(100,truncate=False)

+--------------+
|customer_phone|
+--------------+
|8357109965    |
|6248578017    |
|9406284028    |
|8249926919    |
|9655742829    |
|6178663096    |
|7750572325    |
|9952266893    |
|6200140141    |
|6359154084    |
|9334978881    |
|8481469012    |
|8816070802    |
|8141078352    |
|9719106699    |
|7960514523    |
|6807577342    |
|8327268503    |
|6334760738    |
|8990566476    |
|6781057736    |
|7480990401    |
|7608482316    |
|7678998028    |
|8749431082    |
|6191969690    |
|6903132646    |
|8379995032    |
|7856528252    |
|9289125618    |
|9810303128    |
|7841536135    |
|7784683755    |
|9390894296    |
|7506919288    |
|7624232414    |
+--------------+



In [0]:
phone_pattern=r"^[6-9][0-9]{9}$"

invalid_phone_df=normalized_phone_df.filter(
    F.col("customer_phone").isNotNull() &
    (F.trim(F.col("customer_phone"))!="") &
    ~F.col("customer_phone").rlike(phone_pattern)
)

invalid_phone_df.select(
    "order_id",
    "customer_id",
    "customer_phone"
).show(truncate=False)

+--------+-----------+--------------+
|order_id|customer_id|customer_phone|
+--------+-----------+--------------+
+--------+-----------+--------------+



### Identifier Validation

In [0]:
# Check for missing identifiers
incremental_bronze_orders.select(
    F.sum((F.col("order_id").isNull()).cast("int")).alias("null_order_id"),
    F.sum((F.col("customer_id").isNull()).cast("int")).alias("null_customer_id"),
    F.sum((F.col("product_id").isNull()).cast("int")).alias("null_product_id")
).show()

+-------------+----------------+---------------+
|null_order_id|null_customer_id|null_product_id|
+-------------+----------------+---------------+
|            0|               0|              0|
+-------------+----------------+---------------+



In [0]:
incremental_bronze_orders.select(
    F.sum((F.trim(F.col("order_id"))=="").cast("int")).alias("blank_order_id"),
    F.sum((F.trim(F.col("customer_id")) == "").cast("int")).alias("blank_customer_id"),
    F.sum((F.trim(F.col("product_id"))=="").cast("int")).alias("blank_product_id")
    
).show()

+--------------+-----------------+----------------+
|blank_order_id|blank_customer_id|blank_product_id|
+--------------+-----------------+----------------+
|             0|                0|               0|
+--------------+-----------------+----------------+



### Validate order_id

In [0]:
order_id_pattern=r"^ORD[0-9]{6}$"
invalid_order_id_df=incremental_bronze_orders.filter((
    ~F.col("order_id").rlike(order_id_pattern)
))

invalid_order_id_df.select("order_id").show(truncate=False)

+--------+
|order_id|
+--------+
+--------+



### Validate customer_id

In [0]:
customer_id_pattern=r"^CUST[0-9]{5}$"
invalid_customer_id_df=incremental_bronze_orders.filter(
    ~F.col("customer_id").rlike(customer_id_pattern)
)
invalid_customer_id_df.select("customer_id").show(truncate=False)

+-----------+
|customer_id|
+-----------+
+-----------+



### Validate product_id

In [0]:
product_id_pattern=r"^PROD[0-9]{4}$"

invalid_product_id_df=incremental_bronze_orders.filter(
    ~F.col("product_id").rlike(product_id_pattern)
)

invalid_product_id_df.select("product_id").show()

+----------+
|product_id|
+----------+
+----------+



### Check for Missing Product Names

In [0]:
incremental_bronze_orders.filter(
    F.col("product_name").isNull()
).select(
    "order_id",
    "product_id",
    "product_name",
    "category",
    "order_status"
).show(truncate=False)

+--------+----------+------------+--------+------------+
|order_id|product_id|product_name|category|order_status|
+--------+----------+------------+--------+------------+
+--------+----------+------------+--------+------------+



In [0]:
incremental_bronze_orders.filter(
    F.col("product_name").isNotNull() &
    (F.trim(F.col("product_name")) == "")
).select(
    "order_id",
    "product_id",
    "product_name"
).show()

+--------+----------+------------+
|order_id|product_id|product_name|
+--------+----------+------------+
+--------+----------+------------+



In [0]:
incremental_bronze_orders.groupBy(
    "product_id"
    ).agg(F.countDistinct(
        F.trim(F.col("product_id"))).alias("distinct_product_id")).orderBy("product_id").show()

+----------+-------------------+
|product_id|distinct_product_id|
+----------+-------------------+
|  PROD1002|                  1|
|  PROD1009|                  1|
|  PROD1010|                  1|
|  PROD1018|                  1|
|  PROD1028|                  1|
|  PROD1030|                  1|
|  PROD1031|                  1|
|  PROD1033|                  1|
|  PROD1034|                  1|
|  PROD1035|                  1|
|  PROD1036|                  1|
|  PROD1038|                  1|
|  PROD1047|                  1|
|  PROD1049|                  1|
|  PROD1051|                  1|
|  PROD1054|                  1|
|  PROD1056|                  1|
|  PROD1059|                  1|
|  PROD1064|                  1|
|  PROD1071|                  1|
+----------+-------------------+
only showing top 20 rows


### Create a Product Reference

In [0]:
product_reference = (
    incremental_bronze_orders.filter(
        F.col("product_id").isNotNull() &
        (F.trim(F.col("product_name"))!="")
    )
    .select(
        "product_id",
        F.trim(F.col("product_name")).alias("reference_product_name")
    ).dropDuplicates(["product_id"])
)

In [0]:
product_reference.show(product_reference.count())

+----------+----------------------+
|product_id|reference_product_name|
+----------+----------------------+
|  PROD1076|         Running Shoes|
|  PROD1082|            Hair Serum|
|  PROD1010|         Running Shoes|
|  PROD1071|           LED Monitor|
|  PROD1089|   Wireless Headphones|
|  PROD1038|            Hair Serum|
|  PROD1059|      Herbal Face Wash|
|  PROD1100|         Mixer Grinder|
|  PROD1035|          Office Chair|
|  PROD1074|       Laptop Backpack|
|  PROD1049|           LED Monitor|
|  PROD1030|       Laptop Backpack|
|  PROD1054|         Running Shoes|
|  PROD1091|           Smart Watch|
|  PROD1047|           Smart Watch|
|  PROD1075|          Cotton Shirt|
|  PROD1051|                 Mouse|
|  PROD1034|         Mixer Grinder|
|  PROD1036|          Water Bottle|
|  PROD1083|     Organic Rice Pack|
|  PROD1064|           Cricket Bat|
|  PROD1077|              Backpack|
|  PROD1002|            Smartphone|
|  PROD1031|          Cotton Shirt|
|  PROD1028|              Ke

### Join the Reference Back to the Orders

In [0]:
orders_with_product_reference = (
    incremental_bronze_orders.join(
    product_reference,
    on="product_id",
    how="left"
)
)

In [0]:
orders_with_product_reference.filter(F.col("product_name").isNull()).show()

+----------+--------+-----------+-------------+--------------+--------------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+----------------------+
|product_id|order_id|customer_id|customer_name|customer_email|customer_phone|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|reference_product_name|
+----------+--------+-----------+-------------+--------------+--------------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+----------

In [0]:
orders_with_product_name = (
    orders_with_product_reference.withColumn(
        "product_name",
        F.when(
            F.col("product_name").isNull() |
            (F.trim(F.col("product_name"))==""),
            F.col("reference_product_name")
        ).otherwise(
            F.trim(F.col("product_name"))
        )
    ).drop("reference_product_name")
)
orders_with_product_name.show()

+----------+---------+-----------+---------------+--------------------+--------------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
|product_id| order_id|customer_id|  customer_name|      customer_email|customer_phone|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+----------+---------+-----------+---------------+--------------------+--------------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--------

### Verify the Result

In [0]:
orders_with_product_name.filter(
    F.col("product_name").isNull() |
    (F.trim(F.col("product_name"))== "")
).select(
    "order_id",
    "product_id",
    "product_name"
).show()

+--------+----------+------------+
|order_id|product_id|product_name|
+--------+----------+------------+
+--------+----------+------------+



In [0]:
category_null=incremental_bronze_orders.select(
    F.sum((F.col("category").isNull()).cast("int")).alias("null_category")
)
category_null.show()

+-------------+
|null_category|
+-------------+
|            0|
+-------------+



### Discover the Existing Categories

In [0]:
incremental_bronze_orders.select("category").distinct().show()

+-----------+
|   category|
+-----------+
|    Fashion|
|     Beauty|
|Electronics|
|       Home|
|    Grocery|
|     Sports|
|      Books|
+-----------+



### Analyze Category Frequency

In [0]:
incremental_bronze_orders.groupBy("category").count().orderBy(F.desc("count")).show(truncate=False)


+-----------+-----+
|category   |count|
+-----------+-----+
|Electronics|19   |
|Fashion    |11   |
|Home       |6    |
|Beauty     |5    |
|Grocery    |2    |
|Sports     |1    |
|Books      |1    |
+-----------+-----+



### Check for Missing or Blank Categories

In [0]:
incremental_bronze_orders.filter(
    F.col("category").isNull() |
    (F.trim(F.col("category"))=="")
).select("order_id","product_id","product_name","category").show(truncate=False)

+--------+----------+------------+--------+
|order_id|product_id|product_name|category|
+--------+----------+------------+--------+
+--------+----------+------------+--------+



### Normalize the Category for Investigation

In [0]:
category_analysis=(
    incremental_bronze_orders.withColumn(
        "normalized_category",
        F.upper(F.trim(F.col("category")))
    )
)
category_analysis.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-------------------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|normalized_category|
+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+---

In [0]:
category_analysis.groupBy("normalized_category").count().orderBy(F.desc("count")).show()

+-------------------+-----+
|normalized_category|count|
+-------------------+-----+
|        ELECTRONICS|   19|
|            FASHION|   11|
|               HOME|    6|
|             BEAUTY|    5|
|            GROCERY|    2|
|             SPORTS|    1|
|              BOOKS|    1|
+-------------------+-----+



### Standardize Valid Variations

In [0]:
orders_with_category=incremental_bronze_orders.withColumn(
    "category",
    F.upper(F.trim(F.col("category")))
)
orders_with_category.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--------

In [0]:
valid_category=[
    "ELECTRONICS",
    "HOME",
    "FASHION",
    "BEAUTY",
    "BOOKS",
    "GROCERY",
    "SPORTS"
]

In [0]:
invalid_category_df=(
    orders_with_category.filter(
        F.col("category").isNotNull() &
        ~F.col("category").isin(valid_category)
    )
)
invalid_category_df.select(
    "order_id",
    "product_id",
    "product_name",
    "category"
).show()

+--------+----------+------------+--------+
|order_id|product_id|product_name|category|
+--------+----------+------------+--------+
+--------+----------+------------+--------+



### Investigate the Quantity Column

In [0]:
incremental_bronze_orders.select(
    "order_id",
    "product_id",
    "quantity"
).show(20)

+---------+----------+--------+
| order_id|product_id|quantity|
+---------+----------+--------+
|ORD100208|  PROD1076|       7|
|ORD100244|  PROD1082|       8|
|ORD100779|  PROD1010|       5|
|ORD100778|  PROD1071|       5|
|ORD100428|  PROD1089|       1|
|ORD100923|  PROD1038|       3|
|ORD100111|  PROD1059|       4|
|ORD100958|  PROD1071|       5|
|ORD100959|  PROD1100|       2|
|ORD100960|  PROD1082|       1|
|ORD100011|  PROD1035|       9|
|ORD100365|  PROD1074|       1|
|ORD100954|  PROD1049|       1|
|ORD100953|  PROD1030|       3|
|ORD100382|  PROD1054|      10|
|ORD100099|  PROD1091|       2|
|ORD100077|  PROD1047|       4|
|ORD100298|  PROD1100|       6|
|ORD100963|  PROD1074|       5|
|ORD100313|  PROD1075|      10|
+---------+----------+--------+
only showing top 20 rows


In [0]:
incremental_bronze_orders.filter(F.col("quantity").isNull()).select(
    "order_id",
    "product_id",
    "quantity"
).show()

+--------+----------+--------+
|order_id|product_id|quantity|
+--------+----------+--------+
+--------+----------+--------+



In [0]:
invalid_quantity_df=incremental_bronze_orders.filter(
    F.col("quantity").isNull() |
    (F.col("quantity")<=0)
)
invalid_quantity_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

### Profile the Quantity Distribution

In [0]:
incremental_bronze_orders.select(
    F.min("quantity").alias("min quantity"),
    F.max("quantity").alias("max quantity"),
    F.avg("quantity").alias("average quantity")
).show()

+------------+------------+-----------------+
|min quantity|max quantity| average quantity|
+------------+------------+-----------------+
|           1|          10|4.644444444444445|
+------------+------------+-----------------+



In [0]:
quantity_quality_summary=incremental_bronze_orders.select(
    F.sum((F.col("quantity").isNull()).cast("int")).alias("Null quantity"),
    F.sum((F.col("quantity")==0).cast("int")).alias("Zero quantity"),
    F.sum((F.col("quantity")<0).cast("int")).alias("Negative quantity"),
    F.sum((F.col("quantity")>0).cast("int")).alias("Positive quantity")
)
quantity_quality_summary.show()

+-------------+-------------+-----------------+-----------------+
|Null quantity|Zero quantity|Negative quantity|Positive quantity|
+-------------+-------------+-----------------+-----------------+
|            0|            0|                0|               45|
+-------------+-------------+-----------------+-----------------+



### Investigate the Column

In [0]:
incremental_bronze_orders.select(
    "order_id",
    "product_id",
    "product_name",
    "quantity",
    "unit_price"
).show()

+---------+----------+-------------------+--------+----------+
| order_id|product_id|       product_name|quantity|unit_price|
+---------+----------+-------------------+--------+----------+
|ORD100208|  PROD1076|      Running Shoes|       7|   3590.44|
|ORD100244|  PROD1082|         Hair Serum|       8|   1043.93|
|ORD100779|  PROD1010|      Running Shoes|       5|   7968.44|
|ORD100778|  PROD1071|        LED Monitor|       5|   16166.3|
|ORD100428|  PROD1089|Wireless Headphones|       1|   3511.57|
|ORD100923|  PROD1038|         Hair Serum|       3|    987.96|
|ORD100111|  PROD1059|   Herbal Face Wash|       4|    287.81|
|ORD100958|  PROD1071|        LED Monitor|       5|  14302.16|
|ORD100959|  PROD1100|      Mixer Grinder|       2|  10581.53|
|ORD100960|  PROD1082|         Hair Serum|       1|    321.12|
|ORD100011|  PROD1035|       Office Chair|       9|   15109.6|
|ORD100365|  PROD1074|    Laptop Backpack|       1|   1839.99|
|ORD100954|  PROD1049|        LED Monitor|       1|  32

In [0]:
incremental_bronze_orders.filter(F.col("unit_price").isNull()).select(
    "order_id",
    "product_id",
    "product_name",
    "quantity",
    "unit_price"
).show()

+--------+----------+------------+--------+----------+
|order_id|product_id|product_name|quantity|unit_price|
+--------+----------+------------+--------+----------+
+--------+----------+------------+--------+----------+



In [0]:
incremental_bronze_orders.filter(F.col("unit_price")<=0).select(
    "order_id",
    "product_id",
    "product_name",
    "quantity",
    "unit_price"
).show()

+--------+----------+------------+--------+----------+
|order_id|product_id|product_name|quantity|unit_price|
+--------+----------+------------+--------+----------+
+--------+----------+------------+--------+----------+



In [0]:
invalid_unit_price_df=incremental_bronze_orders.filter(
    (F.col("unit_price").isNull()) |
    (F.col("unit_price")<=0)
)
invalid_unit_price_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

### Profile the Price Values

In [0]:
incremental_bronze_orders.select(
    F.min("unit_price").alias("Minimum unit price"),
    F.max("unit_price").alias("Maximum unit price"),
    F.avg("unit_price").alias("Average unit price")
).show()

+------------------+------------------+------------------+
|Minimum unit price|Maximum unit price|Average unit price|
+------------------+------------------+------------------+
|            287.81|         105982.42| 10458.90511111111|
+------------------+------------------+------------------+



In [0]:
incremental_bronze_orders.select(
    F.sum((F.col("unit_price").isNull()).cast("int")).alias("Null unit price"),
    F.sum((F.col("unit_price")==0).cast("int")).alias("Zero unit price"),
    F.sum((F.col("unit_price")>0).cast("int")).alias("Postive unit price"),
    F.sum((F.col("unit_price")<0).cast("int")).alias("Negative unit price")
).show()

+---------------+---------------+------------------+-------------------+
|Null unit price|Zero unit price|Postive unit price|Negative unit price|
+---------------+---------------+------------------+-------------------+
|              0|              0|                45|                  0|
+---------------+---------------+------------------+-------------------+



### Investigate the Column

In [0]:
incremental_bronze_orders.select(
    "order_id",
    "quantity",
    "unit_price",
    "discount_percent"
).show()

+---------+--------+----------+----------------+
| order_id|quantity|unit_price|discount_percent|
+---------+--------+----------+----------------+
|ORD100208|       7|   3590.44|              15|
|ORD100244|       8|   1043.93|               0|
|ORD100779|       5|   7968.44|              50|
|ORD100778|       5|   16166.3|              40|
|ORD100428|       1|   3511.57|              40|
|ORD100923|       3|    987.96|             100|
|ORD100111|       4|    287.81|              10|
|ORD100958|       5|  14302.16|              15|
|ORD100959|       2|  10581.53|               0|
|ORD100960|       1|    321.12|               5|
|ORD100011|       9|   15109.6|              20|
|ORD100365|       1|   1839.99|              15|
|ORD100954|       1|  32296.61|              10|
|ORD100953|       3|    1960.7|              15|
|ORD100382|      10|   5087.14|              80|
|ORD100099|       2|   19924.9|              10|
|ORD100077|       4|  33981.22|              30|
|ORD100298|       6|

In [0]:
incremental_bronze_orders.select("discount_percent").distinct().show()

+----------------+
|discount_percent|
+----------------+
|              15|
|               0|
|              50|
|              40|
|             100|
|              10|
|               5|
|              20|
|              80|
|              30|
|              60|
|              25|
+----------------+



In [0]:
incremental_bronze_orders.filter(
    F.col("discount_percent").isNull()
    ).select(
        "order_id",
        "product_id",
        "discount_percent"
    ).show()

+--------+----------+----------------+
|order_id|product_id|discount_percent|
+--------+----------+----------------+
+--------+----------+----------------+



### Identify Invalid Discounts

In [0]:
invalid_discount_df=(
    incremental_bronze_orders.filter(
        F.col("discount_percent").isNotNull() & (
            (F.col("discount_percent")<0) |
            (F.col("discount_percent")>80)
        )
    )
)
invalid_discount_df.show()

+---------+-----------+-------------+--------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|customer_name|      customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-------------+--------------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------

### Inspect the Existing Values

In [0]:
incremental_bronze_orders.select(
    "order_id",
    "quantity",
    "unit_price",
    "order_amount"
).show()

+---------+--------+----------+------------+
| order_id|quantity|unit_price|order_amount|
+---------+--------+----------+------------+
|ORD100208|       7|   3590.44|    25133.08|
|ORD100244|       8|   1043.93|     8351.44|
|ORD100779|       5|   7968.44|     39842.2|
|ORD100778|       5|   16166.3|     80831.5|
|ORD100428|       1|   3511.57|     3511.57|
|ORD100923|       3|    987.96|     2963.88|
|ORD100111|       4|    287.81|     1151.24|
|ORD100958|       5|  14302.16|     71510.8|
|ORD100959|       2|  10581.53|    21163.06|
|ORD100960|       1|    321.12|      321.12|
|ORD100011|       9|   15109.6|    135986.4|
|ORD100365|       1|   1839.99|     1839.99|
|ORD100954|       1|  32296.61|    32296.61|
|ORD100953|       3|    1960.7|      5882.1|
|ORD100382|      10|   5087.14|     50871.4|
|ORD100099|       2|   19924.9|     39849.8|
|ORD100077|       4|  33981.22|   135924.88|
|ORD100298|       6|   8035.86|    48215.16|
|ORD100963|       5|   3242.49|    16212.45|
|ORD100313

### Calculate the Expected Order Amount

In [0]:
order_amount_check_df=incremental_bronze_orders.withColumn(
    "expected_order_amount",
    F.round(
        F.col("quantity")*F.col("unit_price"),
        2
        )
)
order_amount_check_df.select(
    "order_id",
    "quantity",
    "unit_price",
    "order_amount",
    "expected_order_amount",
    ).show()

+---------+--------+----------+------------+---------------------+
| order_id|quantity|unit_price|order_amount|expected_order_amount|
+---------+--------+----------+------------+---------------------+
|ORD100208|       7|   3590.44|    25133.08|             25133.08|
|ORD100244|       8|   1043.93|     8351.44|              8351.44|
|ORD100779|       5|   7968.44|     39842.2|              39842.2|
|ORD100778|       5|   16166.3|     80831.5|              80831.5|
|ORD100428|       1|   3511.57|     3511.57|              3511.57|
|ORD100923|       3|    987.96|     2963.88|              2963.88|
|ORD100111|       4|    287.81|     1151.24|              1151.24|
|ORD100958|       5|  14302.16|     71510.8|              71510.8|
|ORD100959|       2|  10581.53|    21163.06|             21163.06|
|ORD100960|       1|    321.12|      321.12|               321.12|
|ORD100011|       9|   15109.6|    135986.4|             135986.4|
|ORD100365|       1|   1839.99|     1839.99|              1839

In [0]:
# why cross-column check we are doing beacuse see the unit price and quantity with order amount and expected order amount
invalid_order_amount_df=order_amount_check_df.filter(
    F.col("order_amount").isNull() |
    F.col("expected_order_amount").isNull() 
)
invalid_order_amount_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+---------------------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|expected_order_amount|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+------------

In [0]:
invalid_order_amount_df=order_amount_check_df.filter(
    F.col("order_amount").isNull() |
    F.col("expected_order_amount").isNull() |
    (
    F.abs(
        F.col("order_amount")-F.col("expected_order_amount")
    )>0.01
    )
)
invalid_order_amount_df.select(
    "order_id",
    "quantity",
    "unit_price",
    "order_amount",
    "expected_order_amount"
).show()

+--------+--------+----------+------------+---------------------+
|order_id|quantity|unit_price|order_amount|expected_order_amount|
+--------+--------+----------+------------+---------------------+
+--------+--------+----------+------------+---------------------+



### Inspect the Existing Values

In [0]:
incremental_bronze_orders.select(
    "order_id",
    "order_amount",
    "discount_percent",
    "final_amount"
).show()

+---------+------------+----------------+------------+
| order_id|order_amount|discount_percent|final_amount|
+---------+------------+----------------+------------+
|ORD100208|    25133.08|              15|    21363.12|
|ORD100244|     8351.44|               0|     8351.44|
|ORD100779|     39842.2|              50|    19977.58|
|ORD100778|     80831.5|              40|     48498.9|
|ORD100428|     3511.57|              40|     2106.94|
|ORD100923|     2963.88|             100|      2371.1|
|ORD100111|     1151.24|              10|     1036.12|
|ORD100958|     71510.8|              15|    60784.18|
|ORD100959|    21163.06|               0|    21163.06|
|ORD100960|      321.12|               5|      305.06|
|ORD100011|    135986.4|              20|   108789.12|
|ORD100365|     1839.99|              15|     1563.99|
|ORD100954|    32296.61|              10|    29066.95|
|ORD100953|      5882.1|              15|     4999.78|
|ORD100382|     50871.4|              80|    10174.28|
|ORD100099

### Calculate the Expected Final Amount

In [0]:
final_amount_check_df=incremental_bronze_orders.withColumn(
    "expected_final_amount",
    F.round(
        (F.col("order_amount")-(F.col("order_amount")*(F.col("discount_percent")/100)))
        ,2
        )
)
final_amount_check_df.select(
    "order_id",
    "order_amount",
    "discount_percent",
    "final_amount",
    "expected_final_amount"
    ).show()

+---------+------------+----------------+------------+---------------------+
| order_id|order_amount|discount_percent|final_amount|expected_final_amount|
+---------+------------+----------------+------------+---------------------+
|ORD100208|    25133.08|              15|    21363.12|             21363.12|
|ORD100244|     8351.44|               0|     8351.44|              8351.44|
|ORD100779|     39842.2|              50|    19977.58|              19921.1|
|ORD100778|     80831.5|              40|     48498.9|              48498.9|
|ORD100428|     3511.57|              40|     2106.94|              2106.94|
|ORD100923|     2963.88|             100|      2371.1|                  0.0|
|ORD100111|     1151.24|              10|     1036.12|              1036.12|
|ORD100958|     71510.8|              15|    60784.18|             60784.18|
|ORD100959|    21163.06|               0|    21163.06|             21163.06|
|ORD100960|      321.12|               5|      305.06|               305.06|

### Identify Mismatches

In [0]:
invalid_final_amount_df=final_amount_check_df.filter(
    F.col("final_amount").isNull() |
    F.col("expected_final_amount").isNull() |
    (
        F.abs(
            F.col("final_amount")-F.col("expected_final_amount")
        )>0.01
    )
)
invalid_final_amount_df.select(
    "order_id",
    "order_amount",
    "discount_percent",
    "final_amount",
    "expected_final_amount"
).show()

+---------+------------+----------------+------------+---------------------+
| order_id|order_amount|discount_percent|final_amount|expected_final_amount|
+---------+------------+----------------+------------+---------------------+
|ORD100779|     39842.2|              50|    19977.58|              19921.1|
|ORD100923|     2963.88|             100|      2371.1|                  0.0|
|ORD100953|      5882.1|              15|     4999.78|              4999.79|
|ORD100220|    103292.7|               5|    98128.06|             98128.07|
+---------+------------+----------------+------------+---------------------+



### Discover Payment Methods

In [0]:
incremental_bronze_orders.select("payment_method").distinct().show(truncate=False)

+----------------+
|payment_method  |
+----------------+
|Debit Card      |
|Wallet          |
|Credit Card     |
|Cash on Delivery|
|UPI             |
|Net Banking     |
+----------------+



### Group and Count Each Payment Method

In [0]:
incremental_bronze_orders.groupBy("payment_method").count().orderBy("count",ascending=False).show()

+----------------+-----+
|  payment_method|count|
+----------------+-----+
|          Wallet|   12|
|             UPI|    8|
|     Credit Card|    8|
|     Net Banking|    7|
|      Debit Card|    6|
|Cash on Delivery|    4|
+----------------+-----+



### Normalize Whitespace and Case

In [0]:
payment_method_analysis=incremental_bronze_orders.withColumn(
    "normalized_payment_method",
    F.upper(
        F.trim(
        F.col("payment_method")
    )
    )
)
payment_method_analysis.groupBy("normalized_payment_method").count().orderBy(F.desc("count")).show()

+-------------------------+-----+
|normalized_payment_method|count|
+-------------------------+-----+
|                   WALLET|   12|
|                      UPI|    8|
|              CREDIT CARD|    8|
|              NET BANKING|    7|
|               DEBIT CARD|    6|
|         CASH ON DELIVERY|    4|
+-------------------------+-----+



### Apply the Standardization

In [0]:
payment_method_mapping={
    'UPI':'UPI',
    'DEBIT CARD':'DEBIT CARD',
    'DEBITCARD':'DEBIT CARD',
    'CREDIT CARD':'CREDIT CARD',
    'CREDITCARD':'CREDIT CARD',
    'COD':'CASH ON DELIVERY',
    'CASH ON DELIVERY':'CASH ON DELIVERY',
    'NET BANKING':'NET BANKING',
    'NETBANKING':'NET BANKING',
    'WALLET':'WALLET'
}

In [0]:
order_with_payment_method=payment_method_analysis.withColumn(
    "payment_method",
        F.create_map(
            *[
                F.lit(x)
           for pair in payment_method_mapping.items()
           for x in pair 
            ]
        )[F.col("normalized_payment_method")]
)
order_with_payment_method.select(
    "payment_method"
).show()

+----------------+
|  payment_method|
+----------------+
|      DEBIT CARD|
|          WALLET|
|          WALLET|
|     CREDIT CARD|
|CASH ON DELIVERY|
|          WALLET|
|             UPI|
|     CREDIT CARD|
|          WALLET|
|CASH ON DELIVERY|
|             UPI|
|CASH ON DELIVERY|
|     CREDIT CARD|
|             UPI|
|             UPI|
|      DEBIT CARD|
|      DEBIT CARD|
|          WALLET|
|     CREDIT CARD|
|     NET BANKING|
+----------------+
only showing top 20 rows


### Discover the Existing Values

In [0]:
incremental_bronze_orders.select("payment_status").distinct().show(truncate=False)

+--------------+
|payment_status|
+--------------+
|Paid          |
|Failed        |
|Refunded      |
|Pending       |
+--------------+



In [0]:
incremental_bronze_orders.groupBy("payment_status").count().orderBy(F.desc("count")).show()

+--------------+-----+
|payment_status|count|
+--------------+-----+
|          Paid|   26|
|       Pending|    8|
|        Failed|    6|
|      Refunded|    5|
+--------------+-----+



In [0]:
payment_status_analysis=incremental_bronze_orders.withColumn(
    "normalized_payment_status",
    F.upper(
        F.trim(
            F.col("payment_status")
            )
    )
)

In [0]:
payment_status_analysis.groupBy("normalized_payment_status").count().orderBy("count",ascending=False).show()

+-------------------------+-----+
|normalized_payment_status|count|
+-------------------------+-----+
|                     PAID|   26|
|                  PENDING|    8|
|                   FAILED|    6|
|                 REFUNDED|    5|
+-------------------------+-----+



In [0]:
payment_status_cleaned=payment_status_analysis.withColumn(
    "payment_status",
    F.when(
        F.col("normalized_payment_status")=="SUCCESS",
        F.lit("PAID")
    ).when(
        F.col("normalized_payment_status") == "REFUNDD",
        F.lit("REFUNDED")
    )
    .otherwise(F.col("normalized_payment_status"))
)
payment_status_cleaned.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-------------------------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|normalized_payment_status|
+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+------

### Verify the Standardization

In [0]:
payment_status_cleaned.groupBy("payment_status").count().orderBy(F.desc("count")).show()

+--------------+-----+
|payment_status|count|
+--------------+-----+
|          PAID|   26|
|       PENDING|    8|
|        FAILED|    6|
|      REFUNDED|    5|
+--------------+-----+



### Discover the Existing Values

In [0]:
incremental_bronze_orders.select(
    "order_status"
).distinct().show()

+------------+
|order_status|
+------------+
|   Delivered|
|   Cancelled|
|   Confirmed|
|      Placed|
+------------+



In [0]:
incremental_bronze_orders.groupBy("order_status").count().orderBy(F.desc("count")).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|   17|
|      Placed|   15|
|   Cancelled|   10|
|   Confirmed|    3|
+------------+-----+



### Normalize Whitespace and Case

In [0]:
order_status_analysis=incremental_bronze_orders.withColumn(
    "normalized_order_status",
    F.initcap(
        F.trim(
            F.col("order_status")
        )
    )
)

In [0]:
order_status_analysis.groupBy("normalized_order_status").count().orderBy("count",ascending=False).show()

+-----------------------+-----+
|normalized_order_status|count|
+-----------------------+-----+
|              Delivered|   17|
|                 Placed|   15|
|              Cancelled|   10|
|              Confirmed|    3|
+-----------------------+-----+



In [0]:
known_order_status=[
    'Delivered',
    'Shipped',
    'Placed',
    'Cancelled',
    'Returned',
    'Confirmed',
    'Deliverd',
    'Shiped',
    'Placd'
]
invalid_order_status_df=order_status_analysis.filter(
    ~F.col("normalized_order_status").isin(known_order_status)
)
invalid_order_status_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+-----------------------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|normalized_order_status|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------

In [0]:
order_status_cleaned=order_status_analysis.withColumn(
    "order_status",
    F.when(
        F.col("normalized_order_status")=="Deliverd",
        F.lit("Delivered")
    )
    .when(
        F.col("normalized_order_status")=="Shiped",
        F.lit("Shipped")
    )
    .when(
        F.col("normalized_order_status")=="Placd",
        F.lit("Placed")
    )
    .otherwise(F.col("normalized_order_status"))
)

In [0]:
order_status_cleaned.groupBy("order_status").count().orderBy(F.desc("count")).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|   17|
|      Placed|   15|
|   Cancelled|   10|
|   Confirmed|    3|
+------------+-----+



### Step 15 — Date Validation and Parsing

### First Understand the Raw Values

In [0]:
incremental_bronze_orders.select(
    "order_id",
    "order_date",
    "ship_date",
    "delivery_date"
).show(truncate=False)

+---------+----------+----------+-------------+
|order_id |order_date|ship_date |delivery_date|
+---------+----------+----------+-------------+
|ORD100208|2025-03-13|2025-03-14|2025-03-16   |
|ORD100244|2025-10-15|NULL      |NULL         |
|ORD100779|2025-04-09|2025-04-11|2025-04-15   |
|ORD100778|2025-07-24|2025-07-27|2025-08-02   |
|ORD100428|2025-01-23|2025-01-24|2025-01-27   |
|ORD100923|2025-03-04|2025-03-05|2025-03-08   |
|ORD100111|2025-01-26|2025-01-30|2025-02-02   |
|ORD100958|2026-01-06|NULL      |NULL         |
|ORD100959|2026-01-05|NULL      |NULL         |
|ORD100960|2026-01-07|NULL      |NULL         |
|ORD100011|2025-11-03|NULL      |NULL         |
|ORD100365|2025-02-24|2025-02-27|2025-03-02   |
|ORD100954|2026-01-05|NULL      |NULL         |
|ORD100953|2026-01-07|NULL      |NULL         |
|ORD100382|2025-02-10|NULL      |NULL         |
|ORD100099|2025-03-22|NULL      |NULL         |
|ORD100077|2025-08-18|NULL      |NULL         |
|ORD100298|2025-03-08|2025-03-12|2025-03

### First Inspect the Date Columns

In [0]:
incremental_bronze_orders.select(
    F.count(
        F.when(F.col("order_date").isNull(),1)
    ).alias("null_order_date"),
    F.count(
        F.when(F.col("ship_date").isNull(),1)
    ).alias("null_ship_date"),
    F.count(
        F.when(F.col("delivery_date").isNull(),1)
    ).alias("null_delivery_date")
).show()

+---------------+--------------+------------------+
|null_order_date|null_ship_date|null_delivery_date|
+---------------+--------------+------------------+
|              0|            27|                27|
+---------------+--------------+------------------+



### Parse order_date

In [0]:
order_date_analysis=incremental_bronze_orders.withColumn(
    "parsed_order_date",
    F.try_to_date
    (
        F.col("order_date"),
         "yyyy-MM-dd"
    )
)

In [0]:
order_date_analysis.select(
    "order_id",
    "order_date",
    "parsed_order_date"
).show(truncate=False)

+---------+----------+-----------------+
|order_id |order_date|parsed_order_date|
+---------+----------+-----------------+
|ORD100208|2025-03-13|2025-03-13       |
|ORD100244|2025-10-15|2025-10-15       |
|ORD100779|2025-04-09|2025-04-09       |
|ORD100778|2025-07-24|2025-07-24       |
|ORD100428|2025-01-23|2025-01-23       |
|ORD100923|2025-03-04|2025-03-04       |
|ORD100111|2025-01-26|2025-01-26       |
|ORD100958|2026-01-06|2026-01-06       |
|ORD100959|2026-01-05|2026-01-05       |
|ORD100960|2026-01-07|2026-01-07       |
|ORD100011|2025-11-03|2025-11-03       |
|ORD100365|2025-02-24|2025-02-24       |
|ORD100954|2026-01-05|2026-01-05       |
|ORD100953|2026-01-07|2026-01-07       |
|ORD100382|2025-02-10|2025-02-10       |
|ORD100099|2025-03-22|2025-03-22       |
|ORD100077|2025-08-18|2025-08-18       |
|ORD100298|2025-03-08|2025-03-08       |
|ORD100963|2026-01-05|2026-01-05       |
|ORD100313|2025-12-12|2025-12-12       |
+---------+----------+-----------------+
only showing top

### Identify Failed Date Parsing

In [0]:
invalid_order_date_df=order_date_analysis.filter(
    F.col("order_date").isNull() |
(  
    (F.trim(F.col("order_date"))!="") &
    F.col("parsed_order_date").isNull()
)
)

In [0]:
invalid_order_date_df.select(
    "order_id",
    "order_date",
    "parsed_order_date"
).show(truncate=False)

+--------+----------+-----------------+
|order_id|order_date|parsed_order_date|
+--------+----------+-----------------+
+--------+----------+-----------------+



### Parse All Three Date Columns

In [0]:
order_date_analysis = (
    incremental_bronze_orders.withColumn(
        "parsed_order_date",
        F.try_to_date(
            F.col("order_date"),
            'yyyy-MM-dd'
        )
    )
    .withColumn(
        "parsed_ship_date",
        F.try_to_date(
            F.col("ship_date"),
            'yyyy-MM-dd'
        )
    )
    .withColumn(
        "parsed_delivery_date",
        F.try_to_date(
            F.col("delivery_date"),
            'yyyy-MM-dd'
        )
    )
)

In [0]:
order_date_analysis.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|   17|
|   Cancelled|   10|
|   Confirmed|    3|
|      Placed|   15|
+------------+-----+



### Identify Invalid ship_date

In [0]:
invalid_ship_date_df=order_date_analysis.filter(
    F.col("ship_date").isNotNull() &
    (F.trim(F.col("ship_date"))!="") &
    F.col("parsed_ship_date").isNull()
)

In [0]:
invalid_ship_date_df.select(
    "order_id",
    "ship_date",
    "parsed_ship_date"
    ).show(truncate=False)

+--------+---------+----------------+
|order_id|ship_date|parsed_ship_date|
+--------+---------+----------------+
+--------+---------+----------------+



In [0]:
invalid_delivery_date_df=order_date_analysis.filter(
    F.col("delivery_date").isNull() &
    (F.trim(F.col("delivery_date")!="")) &
    F.col("parsed_delivery_date").isNull()
)

In [0]:
invalid_delivery_date_df.select(
    "order_id",
    "delivery_date",
    "parsed_delivery_date"
).show()

+--------+-------------+--------------------+
|order_id|delivery_date|parsed_delivery_date|
+--------+-------------+--------------------+
+--------+-------------+--------------------+



### Step 16 — Date Business Rules

### Rule 1 — Ship Date Cannot Be Before Order Date

In [0]:
invalid_ship_date_order_df=order_date_analysis.filter(
    F.col("parsed_order_date").isNotNull() &
    F.col("parsed_ship_date").isNotNull() &
    (F.col("parsed_ship_date") < F.col("parsed_order_date"))
)

In [0]:
invalid_ship_date_order_df.select(
    "order_id",
    "parsed_order_date",
    "parsed_ship_date"
).show()

+--------+-----------------+----------------+
|order_id|parsed_order_date|parsed_ship_date|
+--------+-----------------+----------------+
+--------+-----------------+----------------+



### Rule 2 — Delivery Date Cannot Be Before Ship Date

In [0]:
invalid_delivery_date_ship_df=order_date_analysis.filter(
    F.col("parsed_delivery_date").isNotNull() &
    F.col("parsed_ship_date").isNotNull() &
    (F.col("parsed_delivery_date")<F.col("parsed_ship_date"))
)

In [0]:
invalid_delivery_date_ship_df.select(
    "order_id",
    "parsed_ship_date",
    "parsed_delivery_date"
).show()

+--------+----------------+--------------------+
|order_id|parsed_ship_date|parsed_delivery_date|
+--------+----------------+--------------------+
+--------+----------------+--------------------+



In [0]:
order_status_reference = order_status_cleaned.select(
    "order_id",
    "order_status"
)

In [0]:
order_status_reference.show()

+---------+------------+
| order_id|order_status|
+---------+------------+
|ORD100208|   Delivered|
|ORD100244|   Cancelled|
|ORD100779|   Delivered|
|ORD100778|   Confirmed|
|ORD100428|   Delivered|
|ORD100923|   Delivered|
|ORD100111|   Delivered|
|ORD100958|      Placed|
|ORD100959|      Placed|
|ORD100960|      Placed|
|ORD100011|   Confirmed|
|ORD100365|   Delivered|
|ORD100954|      Placed|
|ORD100953|      Placed|
|ORD100382|   Cancelled|
|ORD100099|   Cancelled|
|ORD100077|   Cancelled|
|ORD100298|   Delivered|
|ORD100963|      Placed|
|ORD100313|   Delivered|
+---------+------------+
only showing top 20 rows


In [0]:
order_date_analysis = (
    order_date_analysis
    .drop("order_status")
    .join(
        order_status_reference,
        on="order_id",
        how="left"
    )
)

In [0]:
order_date_analysis.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|      Placed|   15|
|   Confirmed|    3|
|   Delivered|   17|
|   Cancelled|   10|
+------------+-----+



### Rule 3 — Status-Dependent Dates

In [0]:
invalid_shipped_date_df=order_date_analysis.filter(
    (F.col("order_status")=="Shipped") &
    F.col("parsed_ship_date").isNull()
)

In [0]:
invalid_shipped_date_df.select(
    "order_status",
    "parsed_ship_date"
).show()

+------------+----------------+
|order_status|parsed_ship_date|
+------------+----------------+
+------------+----------------+



### Rule 3.2 — Delivered Order Must Have

In [0]:
invalid_deliver_date_df=order_date_analysis.filter(
    (F.col("order_status")=='Delivered') &
    F.col("parsed_delivery_date").isNull()
)

In [0]:
invalid_deliver_date_df.select(
    "order_status",
    "parsed_delivery_date"
).show()

+------------+--------------------+
|order_status|parsed_delivery_date|
+------------+--------------------+
+------------+--------------------+



### Rule 3.3 — Placed Order Should Not Have Shipping or Delivery Dates

In [0]:
invalid_placed_dates_df=order_date_analysis.filter(
    (F.col("order_status") == "Placed") &
    (
    F.col("parsed_delivery_date").isNotNull()|
    F.col("parsed_ship_date").isNotNull()
    )
)

In [0]:
invalid_placed_dates_df.select(
    "order_id",
    "order_status",
    "parsed_delivery_date",
    "parsed_ship_date"
).show()

+--------+------------+--------------------+----------------+
|order_id|order_status|parsed_delivery_date|parsed_ship_date|
+--------+------------+--------------------+----------------+
+--------+------------+--------------------+----------------+



### Rule 3.4 — Confirmed Order

In [0]:
invalid_confirmed_delivery_df=order_date_analysis.filter(
    (F.col("order_status") == "Confirmed") &
    F.col("parsed_delivery_date").isNotNull()
)

### Rule 3.5 — Returned Order

In [0]:
invalid_returned_date_df=order_date_analysis.filter(
    (F.col("order_status") == "Returned") &
    F.col("parsed_delivery_date").isNull()
)

In [0]:
invalid_returned_date_df.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+-----------------+----------------+--------------------+------------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|parsed_order_date|parsed_ship_date|parsed_delivery_date|order_status|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------

### Step 17 — Payment Status and Order Status Business Rules

### Bring Standardized Payment Status

In [0]:
payment_status_reference = payment_status_cleaned.select(
    "order_id",
    "payment_status"
)

In [0]:
order_date_analysis = (
    order_date_analysis
    .drop("payment_status")
    .join(
        payment_status_reference,
        on="order_id",
        how="left"
    )
)

### First Understand the Combinations

In [0]:
payment_order_status_analysis=(
    order_date_analysis
    .groupBy(
        "payment_status",
        "order_status"
    )
    .count()
    .orderBy(F.desc("count"))
)
payment_order_status_analysis.show()

+--------------+------------+-----+
|payment_status|order_status|count|
+--------------+------------+-----+
|          PAID|   Delivered|   17|
|       PENDING|      Placed|    8|
|          PAID|      Placed|    7|
|        FAILED|   Cancelled|    6|
|      REFUNDED|   Cancelled|    4|
|          PAID|   Confirmed|    2|
|      REFUNDED|   Confirmed|    1|
+--------------+------------+-----+



### Business Rule — Delivered + Failed

In [0]:
invalid_delivered_failed_payment_df=order_date_analysis.filter(
    (F.col("order_status") == "Delivered") &
    (F.col("payment_status") == "FAILED")
)

In [0]:
invalid_delivered_failed_payment_df.select(
    "order_id",
    "payment_status",
    "order_status"
).show(truncate=False)

+--------+--------------+------------+
|order_id|payment_status|order_status|
+--------+--------------+------------+
+--------+--------------+------------+



### Business Rule — Delivered + Pending

In [0]:
invalid_delivered_pending_payment_df=order_date_analysis.filter(
    (F.col("order_status") == "Delivered") &
    (F.col("payment_status") == "PENDING")
)

In [0]:
invalid_delivered_pending_payment_df.select(
    "order_id",
    "payment_status",
    "order_status"
).show(truncate=False)

+--------+--------------+------------+
|order_id|payment_status|order_status|
+--------+--------------+------------+
+--------+--------------+------------+



### Business Rule — Placed + Refunded

In [0]:
invalid_placed_refunded_df=order_date_analysis.filter(
    (F.col("order_status")=="Placed") &
    (F.col("payment_status")=="REFUNDED")
)

In [0]:
invalid_placed_refunded_df.select(
    "order_id",
    "payment_status",
    "order_status"
).show(truncate=False)

+--------+--------------+------------+
|order_id|payment_status|order_status|
+--------+--------------+------------+
+--------+--------------+------------+



### Create the Business-Rule Flag

In [0]:
order_date_analysis=order_date_analysis.withColumn(
    "payment_order_status_violation",
    F.when(
    (
        (F.col("order_status")== "Delivered") &
        (F.col("payment_status")=="FAILED")
    ) |
    (
        (F.col("order_status")== "Delivered") &
        (F.col("payment_status")=="PENDING")
    ) |
    (
        (F.col("order_status") == "Placed") &
        (F.col("payment_status") == "REFUNDED")
    ),
    F.lit(True)
    ).otherwise(F.lit(False))
)

In [0]:
order_date_analysis.filter(
    F.col("payment_order_status_violation")==True
).select(
    "order_id",
    "payment_status",
    "order_status"
).show()

+--------+--------------+------------+
|order_id|payment_status|order_status|
+--------+--------------+------------+
+--------+--------------+------------+



### Step 18 — Shipping City Validation

### Investigate Missing or Blank Cities

In [0]:
invalid_shipping_city_df=incremental_bronze_orders.filter(
    F.col("shipping_city").isNull() |
    (
        F.col("shipping_city").isNotNull() &
        (F.trim(F.col("shipping_city"))=="")
    )
)

In [0]:
invalid_shipping_city_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+--------+-------------+--------------+-------+
|order_id|shipping_city|shipping_state|pincode|
+--------+-------------+--------------+-------+
+--------+-------------+--------------+-------+



In [0]:
shipping_city_analysis=incremental_bronze_orders.withColumn(
    "shipping_city",
    F.initcap(
        F.trim(F.col("shipping_city"))
    )
)

In [0]:
shipping_city_analysis = (
    incremental_bronze_orders.withColumn(
        "shipping_city",
        F.when(
            F.trim(F.col("shipping_city"))=="",
            None
        ).otherwise(F.initcap(F.trim(F.col("shipping_city"))))
    )
)

### Recheck the Result

In [0]:
shipping_city_analysis.select(
    "order_id",
    "shipping_city"
).show(truncate=False)

+---------+-------------+
|order_id |shipping_city|
+---------+-------------+
|ORD100208|Kochi        |
|ORD100244|Hyderabad    |
|ORD100779|Delhi        |
|ORD100778|Jaipur       |
|ORD100428|Jaipur       |
|ORD100923|Mumbai       |
|ORD100111|Bengaluru    |
|ORD100958|Delhi        |
|ORD100959|Kochi        |
|ORD100960|Ahmedabad    |
|ORD100011|Delhi        |
|ORD100365|Lucknow      |
|ORD100954|Mumbai       |
|ORD100953|Ahmedabad    |
|ORD100382|Ahmedabad    |
|ORD100099|Bengaluru    |
|ORD100077|Lucknow      |
|ORD100298|Coimbatore   |
|ORD100963|Ahmedabad    |
|ORD100313|Madurai      |
+---------+-------------+
only showing top 20 rows


In [0]:
shipping_city_analysis.filter(
    F.col("pincode")=="625099"
).select(
    "shipping_city",
    "shipping_state"
).show()

+-------------+--------------+
|shipping_city|shipping_state|
+-------------+--------------+
+-------------+--------------+



In [0]:
invalid_shipping_city_df=shipping_city_analysis.filter(
    F.col("shipping_city").isNull()
)
invalid_shipping_city_df.select(
    "order_id",
    "shipping_city"
).show(truncate=False)

+--------+-------------+
|order_id|shipping_city|
+--------+-------------+
+--------+-------------+



### Check pincode-to-city consistency

In [0]:
pincode_city_analysis=(
    incremental_bronze_orders.filter(
        F.col("pincode").isNotNull() &
        (F.trim(F.col("pincode"))!="") &
        F.col("shipping_city").isNotNull() &
        (F.trim(F.col("shipping_city"))!="")
    ).select(
        "pincode",
        F.trim(F.col("shipping_city")).alias("shipping_city")
    )
    .groupBy("pincode")
    .agg(
    F.countDistinct("shipping_city").alias("city_count")
    )
    .orderBy(F.desc("city_count"))
)
pincode_city_analysis.show()

+-------+----------+
|pincode|city_count|
+-------+----------+
| 625013|         1|
| 380098|         1|
| 226060|         1|
| 380001|         1|
| 110077|         1|
| 400015|         1|
| 380063|         1|
| 560025|         1|
| 641005|         1|
| 110014|         1|
| 500048|         1|
| 400063|         1|
| 302079|         1|
| 226065|         1|
| 682092|         1|
| 560055|         1|
| 302018|         1|
| 380037|         1|
| 110055|         1|
| 682040|         1|
+-------+----------+
only showing top 20 rows


### Step 19 — Shipping State Validation and Standardization

### Investigate Distinct State Values

In [0]:
shipping_state_analysis = (
    incremental_bronze_orders.select(
        F.trim(F.col("shipping_state")).alias("shipping_state")
    )
    .groupBy("shipping_state")
    .count()
    .orderBy(F.desc("count"))
)
shipping_state_analysis.show(truncate=False)

+--------------+-----+
|shipping_state|count|
+--------------+-----+
|Tamil Nadu    |8    |
|Delhi         |6    |
|Gujarat       |5    |
|Telangana     |5    |
|Karnataka     |5    |
|Rajasthan     |4    |
|Uttar Pradesh |4    |
|Maharashtra   |3    |
|West Bengal   |3    |
|Kerala        |2    |
+--------------+-----+



### Identify Missing States

In [0]:
invalid_shipping_state_df=incremental_bronze_orders.filter(
    F.col("shipping_state").isNull() |
    (F.trim(F.col("shipping_state"))=="")
)
invalid_shipping_state_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+--------+-------------+--------------+-------+
|order_id|shipping_city|shipping_state|pincode|
+--------+-------------+--------------+-------+
+--------+-------------+--------------+-------+



### Standardize State Formatting

In [0]:
shipping_state_analysis= incremental_bronze_orders.withColumn(
    "shipping_state",
    F.when(
        F.trim(F.col("shipping_state"))=="",
        None
    ).otherwise(
        F.initcap(
            F.trim(F.col("shipping_state"))
        )
    )
)

In [0]:
shipping_state_analysis.groupBy("shipping_state").count().show()

+--------------+-----+
|shipping_state|count|
+--------------+-----+
|        Kerala|    2|
|     Telangana|    5|
|         Delhi|    6|
|     Rajasthan|    4|
|   Maharashtra|    3|
|     Karnataka|    5|
|       Gujarat|    5|
| Uttar Pradesh|    4|
|    Tamil Nadu|    8|
|   West Bengal|    3|
+--------------+-----+



### Step 20 — Pincode Validation

### Identify Missing Pincodes

In [0]:
invalid_pincode_df=incremental_bronze_orders.filter(
    F.col("pincode").isNull() |
    (F.trim(F.col("pincode"))=="")
)
invalid_pincode_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+--------+-------------+--------------+-------+
|order_id|shipping_city|shipping_state|pincode|
+--------+-------------+--------------+-------+
+--------+-------------+--------------+-------+



### Identify Invalid Pincode Formats

In [0]:
pincode_format=r"^[0-9]{6}$"
invalid_pincode_format_df=incremental_bronze_orders.filter(
    F.col("pincode").isNotNull() &
    (F.trim(F.col("pincode"))!="") &
    ~F.trim(F.col("pincode")).rlike(pincode_format)
)

invalid_pincode_format_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode"
).show(truncate=False)

+--------+-------------+--------------+-------+
|order_id|shipping_city|shipping_state|pincode|
+--------+-------------+--------------+-------+
+--------+-------------+--------------+-------+



### Step 21 — Exact Duplicate Analysis

In [0]:
exact_duplicate_analysis = (
    incremental_bronze_orders
    .groupBy(bronze_orders.columns)
    .count()
    .filter(F.col("count")>1)
    .orderBy(F.desc("count"))
)
exact_duplicate_analysis.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+-----+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|count|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-

### Find the Number of Exact Duplicate Records

In [0]:
exact_duplicate_count = (
    exact_duplicate_analysis
    .agg(
        F.sum(F.col("count")-1).alias("duplicate_rows")
    )
)
exact_duplicate_count.show()

+--------------+
|duplicate_rows|
+--------------+
|          NULL|
+--------------+



### Inspect the Exact Duplicates

In [0]:
exact_duplicate_orders = (
    incremental_bronze_orders
    .groupBy(bronze_orders.columns)
    .count()
    .filter(F.col("count")>1)
)

exact_duplicate_orders.select(
    "order_id",
    "customer_id",
    "customer_name",
    "product_id",
    "quantity",
    "unit_price",
    "order_amount",
    "final_amount",
    "order_status",
    "count"
).show(exact_duplicate_orders.count(),truncate=False)

+--------+-----------+-------------+----------+--------+----------+------------+------------+------------+-----+
|order_id|customer_id|customer_name|product_id|quantity|unit_price|order_amount|final_amount|order_status|count|
+--------+-----------+-------------+----------+--------+----------+------------+------------+------------+-----+
+--------+-----------+-------------+----------+--------+----------+------------+------------+------------+-----+



### Step 22 — Same order_id With Different Records

In [0]:
order_id_duplicate_analysis = (
    incremental_bronze_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count")>1)
    .orderBy(F.desc("count"))
)
order_id_duplicate_analysis.show(truncate=False)

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



### Inspect the Duplicate Ordersb

In [0]:
duplicate_orders_ids=order_id_duplicate_analysis.select("order_id")

duplicate_order_records = (
    incremental_bronze_orders
    .join(
        duplicate_orders_ids,
        on="order_id",
        how="inner"
    )
    .orderBy(F.desc("order_id"))
)
# print(duplicate_order_records.count())
duplicate_order_records.show(duplicate_order_records.count(),truncate=False)

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+
|order_id|customer_id|customer_name|customer_email|customer_phone|product_id|product_name|category|quantity|unit_price|discount_percent|order_amount|final_amount|payment_method|payment_status|order_status|order_date|ship_date|delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp|source_file|batch_id|source_system|pipeline_id|
+--------+-----------+-------------+--------------+--------------+----------+------------+--------+--------+----------+----------------+------------+------------+--------------+--------------+------------+----------+---------+-------------+-------------+--------------+-------+-------------------+-----------+-

### Check Whether the Difference Is Real

In [0]:
order_id_variation_analysis = (
    incremental_bronze_orders
    .groupBy("order_id")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("customer_id").alias("customer_id_count"),
        F.countDistinct("product_id").alias("product_id_count"),
        F.countDistinct("quantity").alias("quantity_count"),
        F.countDistinct("unit_price").alias("unit_price_count"),
        F.countDistinct("discount_percent").alias("discount_count"),
        F.countDistinct("order_amount").alias("order_amount_count"),
        F.countDistinct("final_amount").alias("final_amount_count"),
        F.countDistinct("order_status").alias("order_status_count"),
    )
)

order_id_variation_analysis.show(truncate=False)

+---------+------------+-----------------+----------------+--------------+----------------+--------------+------------------+------------------+------------------+
|order_id |record_count|customer_id_count|product_id_count|quantity_count|unit_price_count|discount_count|order_amount_count|final_amount_count|order_status_count|
+---------+------------+-----------------+----------------+--------------+----------------+--------------+------------------+------------------+------------------+
|ORD100208|1           |1                |1               |1             |1               |1             |1                 |1                 |1                 |
|ORD100244|1           |1                |1               |1             |1               |1             |1                 |1                 |1                 |
|ORD100779|1           |1                |1               |1             |1               |1             |1                 |1                 |1                 |
|ORD100778|1    

### Step 23 — Resolve Conflicting order_id Records

### Create a Completeness Score

In [0]:
duplicate_resolution_analysis = (
    incremental_bronze_orders
    .withColumn(
        "completeness_score",
        sum(
            F.when(
                F.col(column).isNotNull() &
                (F.trim(F.col(column).cast("string"))!=""),
                1
            ).otherwise(0)
            for column in bronze_orders.columns
        )
    )
)
duplicate_resolution_analysis.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+------------------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|completeness_score|
+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+-----

### Rank Records Within Each order_id

In [0]:
from pyspark.sql.window import Window

order_window = Window.partitionBy(
    "order_id"
).orderBy(F.desc("completeness_score"))

duplicate_resolution_analysis = (
    duplicate_resolution_analysis
    .withColumn(
        "record_rank",
        F.row_number().over(order_window)
    )
)
duplicate_resolution_analysis.show(truncate=False)

+---------+-----------+-----------------+-----------------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------------+---------------+--------+-------------+-----------+------------------+-----------+
|order_id |customer_id|customer_name    |customer_email               |customer_phone|product_id|product_name       |category   |quantity|unit_price|discount_percent|order_amount|final_amount|payment_method  |payment_status|order_status|order_date|ship_date |delivery_date|shipping_city|shipping_state|pincode|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|completeness_score|record_rank|
+---------+-----------+-----------------+-----------------------------+--------------+----------+-------------------+-----------+--------+----------+-----------

### Step 24 — Identify Multi-Error Records

### Create a Validation Flag for Each Rule

In [0]:
orders_quality=incremental_bronze_orders.withColumn(
    "quantity_invalid",
    F.when(
        F.col("quantity").isNull() |
        (F.col("quantity") <=0),
        True
    ).otherwise(False)
)

In [0]:
orders_quality = orders_quality.withColumn(
    "unit_price_invalid",
    F.when(
        F.col("unit_price").isNull() |
        (F.col("unit_price")<=0),
        True
    ).otherwise(False)
)

### Step 24 — Create Validation Flags.

In [0]:
validation_df=incremental_bronze_orders

In [0]:
validation_df.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--------

In [0]:
# from pyspark.sql import Window
# from pyspark.sql import functions as F

# def dedupe_by_order_id(df, tiebreak_col="order_id"):
#     """
#     Keeps exactly one row per order_id.
#     Replace the orderBy() below with your real survivorship rule.
#     """
#     window_spec = Window.partitionBy("order_id").orderBy(
#         F.monotonically_increasing_id().desc()   # <-- "keep the last-seen row"; change this
#     )
#     return (
#         df
#         .withColumn("_rn", F.row_number().over(window_spec))
#         .filter(F.col("_rn") == 1)
#         .drop("_rn")
#     )

In [0]:
# payment_status_cleaned_deduped   = dedupe_by_order_id(payment_status_cleaned)
# order_status_cleaned_deduped     = dedupe_by_order_id(order_status_cleaned)
# order_with_payment_method_deduped = dedupe_by_order_id(order_with_payment_method)
# orders_with_category_deduped     = dedupe_by_order_id(orders_with_category)

In [0]:
validation_df = (
    validation_df
    .drop("payment_status","order_status","payment_method","category","shipping_city")
    .join(
        payment_status_cleaned
        .select("order_id", "payment_status")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
    .join(
        order_status_cleaned
        .select("order_id", "order_status")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
    .join(
        order_with_payment_method
        .select("order_id", "payment_method")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
    .join(
        orders_with_category
        .select("order_id", "category")
        .dropDuplicates(['order_id']),
        on="order_id", how="left"
    )
    .join(
        shipping_city_analysis
        .select("order_id","shipping_city")
        .dropDuplicates(['order_id']),
        on="order_id",
        how="left"
    )
)

In [0]:
# validation_df = (
#     validation_df
#     .drop("order_status")
#     .join(
#         order_status_cleaned_deduped.select("order_id", "order_status"),
#         on="order_id", how="left"
#     )
# )

In [0]:
# validation_df = (
#     validation_df
#     .drop("payment_method")
#     .join(
#         order_with_payment_method_deduped.select("order_id", "payment_method"),
#         on="order_id", how="left"
#     )
# )

In [0]:
# validation_df = (
#     validation_df
#     .drop("category")
#     .join(
#         orders_with_category_deduped.select("order_id", "category"),
#         on="order_id", how="left"
#     )
# )

In [0]:
validation_df.count()

45

In [0]:
validation_df.select("category","payment_method","payment_status","order_status",).show()

+-----------+----------------+--------------+------------+
|   category|  payment_method|payment_status|order_status|
+-----------+----------------+--------------+------------+
|    FASHION|      DEBIT CARD|          PAID|   Delivered|
|     BEAUTY|          WALLET|        FAILED|   Cancelled|
|    FASHION|          WALLET|          PAID|   Delivered|
|ELECTRONICS|     CREDIT CARD|      REFUNDED|   Confirmed|
|ELECTRONICS|CASH ON DELIVERY|          PAID|   Delivered|
|     BEAUTY|          WALLET|          PAID|   Delivered|
|     BEAUTY|             UPI|          PAID|   Delivered|
|ELECTRONICS|     CREDIT CARD|       PENDING|      Placed|
|       HOME|          WALLET|       PENDING|      Placed|
|     BEAUTY|CASH ON DELIVERY|       PENDING|      Placed|
|       HOME|             UPI|          PAID|   Confirmed|
|ELECTRONICS|CASH ON DELIVERY|          PAID|   Delivered|
|ELECTRONICS|     CREDIT CARD|          PAID|      Placed|
|ELECTRONICS|             UPI|          PAID|      Place

### Step 24.2 — Customer Name, Email and Phone Validation Flags

### Invalid Customer Name

In [0]:
validation_df = (
    validation_df
    .join(
        unrecoverable_customer_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_name",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_name":False
    })
)

### Invalid Customer Email

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_email_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_email",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_email":False
    })
)

### Invalid Customer Phone

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_phone_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_phone",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_phone":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "invalid_customer_name",
    "invalid_customer_email",
    "invalid_customer_phone"
).show(truncate=False)

+---------+---------------------+----------------------+----------------------+
|order_id |invalid_customer_name|invalid_customer_email|invalid_customer_phone|
+---------+---------------------+----------------------+----------------------+
|ORD100208|false                |false                 |false                 |
|ORD100244|false                |false                 |false                 |
|ORD100779|false                |false                 |false                 |
|ORD100778|false                |false                 |false                 |
|ORD100428|false                |false                 |false                 |
|ORD100923|false                |false                 |false                 |
|ORD100111|false                |false                 |false                 |
|ORD100958|false                |false                 |false                 |
|ORD100959|false                |false                 |false                 |
|ORD100960|false                |false  

### 24.3 — Order ID, Customer ID and Product ID validation flags

### Invalid Order ID

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_id_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_order_id",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_id":False
    })
)

### Invalid Customer ID

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_customer_id_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_customer_id",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_customer_id":False
    })
)

### Invalid Product ID

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_product_id_df.select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_product_id",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_product_id":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "invalid_order_id",
    "invalid_customer_id",
    "invalid_product_id"
).show()

+---------+-----------+----------+----------------+-------------------+------------------+
| order_id|customer_id|product_id|invalid_order_id|invalid_customer_id|invalid_product_id|
+---------+-----------+----------+----------------+-------------------+------------------+
|ORD100208|  CUST10032|  PROD1076|           false|              false|             false|
|ORD100244|  CUST10112|  PROD1082|           false|              false|             false|
|ORD100779|  CUST10053|  PROD1010|           false|              false|             false|
|ORD100778|  CUST10020|  PROD1071|           false|              false|             false|
|ORD100428|  CUST10037|  PROD1089|           false|              false|             false|
|ORD100923|  CUST10063|  PROD1038|           false|              false|             false|
|ORD100111|  CUST10130|  PROD1059|           false|              false|             false|
|ORD100958|  CUST10098|  PROD1071|           false|              false|             false|

### Step 24.4 — Product name and category validation flags.

### Invalid Product Name

In [0]:
invalid_product_name_df=orders_with_product_name.filter(
    F.col("product_name").isNull() |
    (F.trim(F.col("product_name"))=="")
).select("order_id").dropDuplicates(["order_id"])

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_product_name_df
        .withColumn(
            "invalid_product_name",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_product_name":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "product_id",
    "product_name",
    "invalid_product_name"
).show()

+---------+----------+-------------------+--------------------+
| order_id|product_id|       product_name|invalid_product_name|
+---------+----------+-------------------+--------------------+
|ORD100208|  PROD1076|      Running Shoes|               false|
|ORD100244|  PROD1082|         Hair Serum|               false|
|ORD100779|  PROD1010|      Running Shoes|               false|
|ORD100778|  PROD1071|        LED Monitor|               false|
|ORD100428|  PROD1089|Wireless Headphones|               false|
|ORD100923|  PROD1038|         Hair Serum|               false|
|ORD100111|  PROD1059|   Herbal Face Wash|               false|
|ORD100958|  PROD1071|        LED Monitor|               false|
|ORD100959|  PROD1100|      Mixer Grinder|               false|
|ORD100960|  PROD1082|         Hair Serum|               false|
|ORD100011|  PROD1035|       Office Chair|               false|
|ORD100365|  PROD1074|    Laptop Backpack|               false|
|ORD100954|  PROD1049|        LED Monito

In [0]:
validation_df = (
    validation_df
    .join(
    invalid_category_df
    .select("order_id")
    .dropDuplicates(["order_id"])
    .withColumn("invalid_category",F.lit(True)),
    on="order_id",
    how="left"
    )
    .fillna({
        "invalid_category":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "category",
    "invalid_category"
).show()

+---------+-----------+----------------+
| order_id|   category|invalid_category|
+---------+-----------+----------------+
|ORD100208|    FASHION|           false|
|ORD100244|     BEAUTY|           false|
|ORD100779|    FASHION|           false|
|ORD100778|ELECTRONICS|           false|
|ORD100428|ELECTRONICS|           false|
|ORD100923|     BEAUTY|           false|
|ORD100111|     BEAUTY|           false|
|ORD100958|ELECTRONICS|           false|
|ORD100959|       HOME|           false|
|ORD100960|     BEAUTY|           false|
|ORD100011|       HOME|           false|
|ORD100365|ELECTRONICS|           false|
|ORD100954|ELECTRONICS|           false|
|ORD100953|ELECTRONICS|           false|
|ORD100382|    FASHION|           false|
|ORD100099|ELECTRONICS|           false|
|ORD100077|ELECTRONICS|           false|
|ORD100298|       HOME|           false|
|ORD100963|ELECTRONICS|           false|
|ORD100313|    FASHION|           false|
+---------+-----------+----------------+
only showing top

### Step 24.5 — Quantity, Unit Price, and Discount Validation Flags

### Invalid Quantity Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_quantity_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_quantity",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_quantity":False
    })
)

### Invalid Unit Price Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_unit_price_df
        .select("order_id")
        .dropDuplicates()
        .withColumn("invalid_unit_price",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_unit_price":False
    })
)

### Invalid Discount Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_discount_df.select("order_id")
        .dropDuplicates()
        .withColumn("invalid_discount_percent",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_discount_percent":False
    })
)

### Check the Three Flags

In [0]:
validation_df.select(
    "order_id",
    "quantity",
    "unit_price",
    "discount_percent",
    "invalid_quantity",
    "invalid_unit_price",
    "invalid_discount_percent"
).show()

+---------+--------+----------+----------------+----------------+------------------+------------------------+
| order_id|quantity|unit_price|discount_percent|invalid_quantity|invalid_unit_price|invalid_discount_percent|
+---------+--------+----------+----------------+----------------+------------------+------------------------+
|ORD100208|       7|   3590.44|              15|           false|             false|                   false|
|ORD100244|       8|   1043.93|               0|           false|             false|                   false|
|ORD100779|       5|   7968.44|              50|           false|             false|                   false|
|ORD100778|       5|   16166.3|              40|           false|             false|                   false|
|ORD100428|       1|   3511.57|              40|           false|             false|                   false|
|ORD100923|       3|    987.96|             100|           false|             false|                    true|
|ORD100111

### Step 24.6 — Order Amount and Final Amount Validation Flags

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_amount_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_order_amount",F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_amount":False
    })
)

### Invalid Final Amount Flag

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_final_amount_df
        .select("order_id")
        .dropDuplicates()
        .withColumn("invalid_final_amount",F.lit(True)),
        on="order_id",
        how="left"
)
    .fillna({
        "invalid_final_amount":False
    })
)

In [0]:
validation_df.select(
    "order_amount",
    "order_amount",
    "final_amount",
    "invalid_order_amount",
    "invalid_final_amount"
).show()

+------------+------------+------------+--------------------+--------------------+
|order_amount|order_amount|final_amount|invalid_order_amount|invalid_final_amount|
+------------+------------+------------+--------------------+--------------------+
|    25133.08|    25133.08|    21363.12|               false|               false|
|     8351.44|     8351.44|     8351.44|               false|               false|
|     39842.2|     39842.2|    19977.58|               false|                true|
|     80831.5|     80831.5|     48498.9|               false|               false|
|     3511.57|     3511.57|     2106.94|               false|               false|
|     2963.88|     2963.88|      2371.1|               false|                true|
|     1151.24|     1151.24|     1036.12|               false|               false|
|     71510.8|     71510.8|    60784.18|               false|               false|
|    21163.06|    21163.06|    21163.06|               false|               false|
|   

In [0]:
validation_df.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+----------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|quantity|unit_price|discount_percent|order_amount|final_amount|order_date| ship_date|delivery_date|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|payment_status|order_status|  payment_method|   category|shippi

In [0]:
order_with_payment_method.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+-------------------------+
| order_id|customer_id|  customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|normalized_payment_method|
+---------+-----------+---------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+------

In [0]:
invalid_payment_method_df=(
    order_with_payment_method
    .filter(
        F.col("payment_method").isNull() |
        (F.trim(F.col('payment_method'))=="")
    )
    .select("order_id")
    .dropDuplicates(["order_id"])
)
invalid_payment_method_df.show()

+--------+
|order_id|
+--------+
+--------+



In [0]:
validation_df = (
    validation_df
    .join(
        invalid_payment_method_df
        .withColumn("invalid_payment_method",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_payment_method":False
    })
)

In [0]:
invalid_payment_status_df = (
    payment_status_cleaned.filter(
        ~F.col("payment_status").isin(
            "PAID",
            "REFUNDED",
            "PENDING",
            "FAILED"
        )
    ).select("order_id").dropDuplicates()
)
invalid_payment_status_df.show()

+--------+
|order_id|
+--------+
+--------+



In [0]:
validation_df = (
    validation_df
    .join(
        invalid_payment_status_df
        .withColumn(
            "invalid_payment_status",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_payment_status": False
    })
)

### Invalid Order Status

In [0]:
invalid_order_status_df_final = (
    order_status_cleaned.filter(
        ~F.col("order_status").isin(
            "Shipped",
            "Cancelled",
            "Delivered",
            "Returned",
            "Placed",
            "Confirmed"
        )
    )
    .select("order_id")
    .dropDuplicates()
)
invalid_order_status_df_final.show()

+--------+
|order_id|
+--------+
+--------+



In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_status_df_final
        .withColumn(
            "invalid_order_status",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_status": False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "payment_method",
    "payment_status",
    "order_status",
    "invalid_payment_method",
    "invalid_payment_status",
    "invalid_order_status"
).show(truncate=False)

+---------+----------------+--------------+------------+----------------------+----------------------+--------------------+
|order_id |payment_method  |payment_status|order_status|invalid_payment_method|invalid_payment_status|invalid_order_status|
+---------+----------------+--------------+------------+----------------------+----------------------+--------------------+
|ORD100208|DEBIT CARD      |PAID          |Delivered   |false                 |false                 |false               |
|ORD100244|WALLET          |FAILED        |Cancelled   |false                 |false                 |false               |
|ORD100779|WALLET          |PAID          |Delivered   |false                 |false                 |false               |
|ORD100778|CREDIT CARD     |REFUNDED      |Confirmed   |false                 |false                 |false               |
|ORD100428|CASH ON DELIVERY|PAID          |Delivered   |false                 |false                 |false               |
|ORD1009

### Step 24.8 — Date Validation Flags

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_order_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_order_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_order_date":False
    })
)

### Invalid Ship Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_ship_date_df
        .select("order_id")
        .dropDuplicates(["order_id"])
        .withColumn("invalid_ship_date",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_ship_date":False
    })
)

### Invalid Delivery Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_deliver_date_df.select("order_id").dropDuplicates()
        .withColumn("invalid_delivery_date",F.lit(True)),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_delivery_date":False
    })
)

### Ship Date Before Order Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_ship_date_order_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_ship_date_order",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_ship_date_order": False
    })
)

### Delivery Date Before Ship Date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_delivery_date_ship_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_delivery_date_ship",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_delivery_date_ship": False
    })
)

### 24.8.6 — Status-Dependent Date Rules

### Shipped without ship date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_shipped_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_shipped_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_shipped_date": False
    })
)

### Delivered without delivery date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_deliver_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_deliver_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_deliver_date": False
    })
)

### Placed with shipping/delivery dates

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_placed_dates_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_placed_dates",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_placed_dates": False
    })
)

### Confirmed with invalid delivery date

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_confirmed_delivery_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_confirmed_delivery",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_confirmed_delivery": False
    })
)

### Returned with date issue

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_returned_date_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_returned_date",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_returned_date": False
    })
)

### Check All Date Flags

In [0]:
validation_df.select(
    "order_id",
    "order_date",
    "ship_date",
    "delivery_date",
    "order_status",
    "invalid_order_date",
    "invalid_ship_date",
    "invalid_delivery_date",
    "invalid_ship_date_order",
    "invalid_delivery_date_ship",
    "invalid_shipped_date",
    "invalid_deliver_date",
    "invalid_placed_dates",
    "invalid_confirmed_delivery",
    "invalid_returned_date"
).show(truncate=False)
validation_df.count()

+---------+----------+----------+-------------+------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+
|order_id |order_date|ship_date |delivery_date|order_status|invalid_order_date|invalid_ship_date|invalid_delivery_date|invalid_ship_date_order|invalid_delivery_date_ship|invalid_shipped_date|invalid_deliver_date|invalid_placed_dates|invalid_confirmed_delivery|invalid_returned_date|
+---------+----------+----------+-------------+------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+
|ORD100208|2025-03-13|2025-03-14|2025-03-16   |Delivered   |false             |false            |false                |false                  |false   

45

### Step 24.9 — Payment + Order Status business-rule flags.

### Add the Existing Business-Rule Flag

In [0]:
validation_df = (
    validation_df
    .join(
        order_date_analysis
        .select(
            "order_id",
            "payment_order_status_violation"
        )
        .dropDuplicates(['order_id']),
        on="order_id",
        how="left"
    )
    .fillna({
        "payment_order_status_violation":False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "payment_status",
    "order_status",
    "payment_order_status_violation"
).show()

+---------+--------------+------------+------------------------------+
| order_id|payment_status|order_status|payment_order_status_violation|
+---------+--------------+------------+------------------------------+
|ORD100208|          PAID|   Delivered|                         false|
|ORD100244|        FAILED|   Cancelled|                         false|
|ORD100779|          PAID|   Delivered|                         false|
|ORD100778|      REFUNDED|   Confirmed|                         false|
|ORD100428|          PAID|   Delivered|                         false|
|ORD100923|          PAID|   Delivered|                         false|
|ORD100111|          PAID|   Delivered|                         false|
|ORD100958|       PENDING|      Placed|                         false|
|ORD100959|       PENDING|      Placed|                         false|
|ORD100960|       PENDING|      Placed|                         false|
|ORD100011|          PAID|   Confirmed|                         false|
|ORD10

In [0]:
validation_df.select(
    "order_id",
    "payment_status",
    "order_status",
    "payment_order_status_violation"
).filter(F.col("payment_order_status_violation")==True).show()

+--------+--------------+------------+------------------------------+
|order_id|payment_status|order_status|payment_order_status_violation|
+--------+--------------+------------+------------------------------+
+--------+--------------+------------+------------------------------+



### Step 24.10 — Shipping City, State and Pincode Validation Flags

### Invalid Shipping City

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_shipping_city_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_shipping_city",F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_shipping_city":False
    })
)

### Invalid Shipping State

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_shipping_state_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_shipping_state",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_shipping_state": False
    })
)

### Invalid Pincode

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_pincode_format_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_pincode_format",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_pincode_format": False
    })
)

In [0]:
validation_df = (
    validation_df
    .join(
        invalid_pincode_df
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "invalid_pincode",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "invalid_pincode": False
    })
)

In [0]:
validation_df.select(
    "order_id",
    "shipping_city",
    "shipping_state",
    "pincode",
    "invalid_shipping_city",
    "invalid_shipping_state",
    "invalid_pincode",
    "invalid_pincode_format"
).show(truncate=False)

+---------+-------------+--------------+-------+---------------------+----------------------+---------------+----------------------+
|order_id |shipping_city|shipping_state|pincode|invalid_shipping_city|invalid_shipping_state|invalid_pincode|invalid_pincode_format|
+---------+-------------+--------------+-------+---------------------+----------------------+---------------+----------------------+
|ORD100208|Kochi        |Kerala        |682092 |false                |false                 |false          |false                 |
|ORD100244|Hyderabad    |Telangana     |500048 |false                |false                 |false          |false                 |
|ORD100779|Delhi        |Delhi         |110055 |false                |false                 |false          |false                 |
|ORD100778|Jaipur       |Rajasthan     |302018 |false                |false                 |false          |false                 |
|ORD100428|Jaipur       |Rajasthan     |302079 |false                

### Step 24.11 — Duplicate and Conflicting Order Flags

### Duplicate Order Flag

In [0]:
validation_df = (
    validation_df
    .join(
        duplicate_orders_ids
        .select("order_id")
        .dropDuplicates()
        .withColumn(
            "duplicate_order",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .fillna({
        "duplicate_order": False
    })
)

In [0]:
validation_df.show()

+---------+-----------+---------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+----------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+----------------------

### Update invalid_customer_name

In [0]:
validation_df = validation_df.withColumn(
    "invalid_customer_name",
    F.when(
        F.col("customer_name").isNull() |
        (F.trim(F.col("customer_name")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_customer_name"))
)

### update invalid_customer_email

In [0]:
validation_df = validation_df.withColumn(
    "invalid_customer_email",
    F.when(
        F.col("customer_email").isNull() |
        (F.trim(F.col("customer_email")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_customer_email"))
)

### update invalid_customer_phone

In [0]:
validation_df = validation_df.withColumn(
    "invalid_customer_phone",
    F.when(
        F.col("customer_phone").isNull() |
        (F.trim(F.col("customer_phone")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_customer_phone"))
)

### update invalid_product_name

In [0]:
validation_df = validation_df.withColumn(
    "invalid_product_name",
    F.when(
        F.col("product_name").isNull() |
        (F.trim(F.col("product_name")) == ""),
        F.lit(True)
    )
    .otherwise(F.col("invalid_product_name"))
)

### Step 25 — Create reject_reason

In [0]:
validation_df = validation_df.withColumn(
    "reject_reason",
    F.concat_ws(
        ", ",
        F.when(
            F.col("invalid_customer_name"),
            F.lit("INVALID_CUSTOMER_NAME")
        ),
        F.when(
            F.col("invalid_customer_email"),
            F.lit("INVALID_CUSTOMER_EMAIL")
        ),
        F.when(
            F.col("invalid_customer_phone"),
            F.lit("INVALID_CUSTOMER_PHONE")
        ),
        F.when(
            F.col("invalid_order_id"),
            F.lit("INVALID_ORDER_ID")
        ),
        F.when(
            F.col("invalid_customer_id"),
            F.lit("INVALID_CUSTOMER_ID")
        ),
        F.when(
            F.col("invalid_product_name"),
            F.lit("INVALID_PRODUCT_NAME")
        ),
        F.when(
            F.col("invalid_category"),
            F.lit("INVALID_CATEGORY")
        ),
        F.when(
            F.col("invalid_quantity"),
            F.lit("INVALIID_QUANTITY")
        ),
        F.when(
            F.col("invalid_unit_price"),
            F.lit("INVALID_UNIT_PRICE")
        ),
        F.when(
            F.col("invalid_discount_percent"),
            F.lit("INVALID_DISCOUNT_PERCENT")
        ),
        F.when(
            F.col("invalid_order_amount"),
            F.lit("INVALID_ORDER_AMOUNT")
        ),
        F.when(
            F.col("invalid_final_amount"),
            F.lit("INVALID_FINAL_AMOUNT")
        ),
        F.when(
            F.col("invalid_payment_method"),
            F.lit("INVALID_PAYMENT_METHOD")
        ),
        F.when(
            F.col("invalid_payment_status"),
            F.lit("INVALID_PAYMENT_STATUS")
        ),
        F.when(
            F.col("invalid_order_status"),
            F.lit("INVALID_ORDER_STATUS")
        ),
        F.when(
            F.col("payment_order_status_violation"),
            F.lit("PAYMENT_ORDER_STATUS_VIOLATION")
        ),
        F.when(
            F.col("invalid_order_date"),
            F.lit("INVALID_ORDER_DATE")
        ),
        F.when(
            F.col("invalid_ship_date"),
            F.lit("INVALID_SHIP_DATE")
        ),
        F.when(
            F.col("invalid_deliver_date"),
            F.lit("INVALID_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_ship_date_order"),
            F.lit("SHIP_DATE_BEFORE_ORDER_DATE")
        ),
        F.when(
            F.col("invalid_delivery_date_ship"),
            F.lit("DELIVERY_DATE_BEFORE_SHIP_DATE")
        ),
        F.when(
            F.col("invalid_ship_date"),
            F.lit("SHIPPED_WITHOUT_SHIP_DATE")
        ),
        F.when(
            F.col("invalid_delivery_date"),
            F.lit("DELIVERY_WITHOUT_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_placed_dates"),
            F.lit("PLACED_WITH_SHIPPING_OR_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_confirmed_delivery"),
            F.lit("CONFIRMED_WITH_INVALID_DELIVERY_DATE")
        ),
        F.when(
            F.col("invalid_returned_date"),
            F.lit("RETURNED_WITH_DATE_ISSUE")
        ),
        F.when(
            F.col("invalid_shipping_city"),
            F.lit("INVALID_SHIPPING_CITY")
        ),
        F.when(
            F.col("invalid_shipping_state"),
            F.lit("INVALID_SHIPPING_STATE")
        ),
        F.when(
            F.col("invalid_pincode"),
            F.lit("INVALID_PINCODE")
        ),
        F.when(
            F.col("invalid_pincode_format"),
            F.lit("INVALID_PINCODE_FORMAT")
        ),
        F.when(
            F.col("duplicate_order"),
            F.lit("DUPLICATE_ORDER")
        ),
    )
)

In [0]:
validation_df.filter(
    F.col("reject_reason")!=""
).show()

+---------+-----------+-------------+--------------------+--------------+----------+---------------+--------+----------+----------------+------------+------------+----------+----------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+--------------+-----------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+----------------------+-------

### Step 26 — Create the Final record_valid Flag

In [0]:
validation_df = validation_df.withColumn(
    "record_valid",
    F.when(
        F.col("reject_reason")=="",
        F.lit(True)
    ).otherwise(F.lit(False))
)

In [0]:
validation_df.select(
    "order_id",
    "reject_reason",
    "record_valid"
).show(truncate=False)

+---------+----------------------------------------------+------------+
|order_id |reject_reason                                 |record_valid|
+---------+----------------------------------------------+------------+
|ORD100208|                                              |true        |
|ORD100244|                                              |true        |
|ORD100779|INVALID_FINAL_AMOUNT                          |false       |
|ORD100778|CONFIRMED_WITH_INVALID_DELIVERY_DATE          |false       |
|ORD100428|                                              |true        |
|ORD100923|INVALID_DISCOUNT_PERCENT, INVALID_FINAL_AMOUNT|false       |
|ORD100111|                                              |true        |
|ORD100958|                                              |true        |
|ORD100959|                                              |true        |
|ORD100960|                                              |true        |
|ORD100011|                                              |true  

### Verify the overall counts

In [0]:
validation_df.groupBy("record_valid").count().show()

+------------+-----+
|record_valid|count|
+------------+-----+
|        true|   39|
|       false|    6|
+------------+-----+



### Step 27 — Split Valid and Invalid Records

### Silver candidate records

In [0]:
incremental_silver_candiate_df = validation_df.filter(
    F.col("record_valid")==True
    )

### Reject records

In [0]:
reject_df = validation_df.filter(
    F.col("record_valid") == False
)

In [0]:
print("Silver Candiated:",incremental_silver_candiate_df.count())
print("Reject records:",reject_df.count())

Silver Candiated: 39
Reject records: 6


In [0]:
print("Bronze:", incremental_bronze_orders.count())
print("Validation:", validation_df.count())
print("Silver:", incremental_silver_candiate_df.count())
print("Reject:", reject_df.count())

Bronze: 45
Validation: 45
Silver: 39
Reject: 6


In [0]:
print(
    "Unique order IDs:",
    validation_df.select("order_id").distinct().count()
)

Unique order IDs: 45


### Step 28 — Write Silver and Reject Data

### Write the Reject Data

In [0]:
reject_df.write.format("delta").mode("append").saveAsTable("ecommerce_project.reject_orders")

### Write the Silver Data

In [0]:
incremental_silver_candiate_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce_project.incremental_silver_orders")

In [0]:
silver_count=spark.read.table("ecommerce_project.incremental_silver_orders").count()
reject_count=spark.read.table("ecommerce_project.reject_orders").count()
print("Silver count:",silver_count)
print("Reject count:",reject_count)

Silver count: 39
Reject count: 281


In [0]:
bronze_count=spark.table("ecommerce_project.bronze_orders").count() - 1000
exiting_silver_count=275
assert silver_count+(reject_count-exiting_silver_count) == bronze_count

print("Bronze records :",bronze_count)
print("Silver records :",silver_count)
print("Reject records :",reject_count)
print("Record count validation :PASS")

Bronze records : 45
Silver records : 39
Reject records : 281
Record count validation :PASS


In [0]:
silver_null_summary = incremental_silver_candiate_df.select(
    *[
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in incremental_silver_candiate_df.columns
    ]
)

In [0]:
silver_null_summary.show()

+--------+-----------+-------------+--------------+--------------+----------+------------+--------+----------+----------------+------------+------------+----------+---------+-------------+--------------+-------+-------------------+-----------+--------+-------------+-----------+--------------+------------+--------------+--------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+----------------------+----------------------+---

In [0]:
%sql
select * from workspace.ecommerce_project.incremental_silver_orders;

order_id,customer_id,customer_name,customer_email,customer_phone,product_id,product_name,quantity,unit_price,discount_percent,order_amount,final_amount,order_date,ship_date,delivery_date,shipping_state,pincode,ingestion_timestamp,source_file,batch_id,source_system,pipeline_id,payment_status,order_status,payment_method,category,shipping_city,invalid_customer_name,invalid_customer_email,invalid_customer_phone,invalid_order_id,invalid_customer_id,invalid_product_id,invalid_product_name,invalid_category,invalid_quantity,invalid_unit_price,invalid_discount_percent,invalid_order_amount,invalid_final_amount,invalid_payment_method,invalid_payment_status,invalid_order_status,invalid_order_date,invalid_ship_date,invalid_delivery_date,invalid_ship_date_order,invalid_delivery_date_ship,invalid_shipped_date,invalid_deliver_date,invalid_placed_dates,invalid_confirmed_delivery,invalid_returned_date,payment_order_status_violation,invalid_shipping_city,invalid_shipping_state,invalid_pincode_format,invalid_pincode,duplicate_order,reject_reason,record_valid
ORD100208,CUST10032,Rahul Pillai,rahul.pillai@outlook.com,8357109965,PROD1076,Running Shoes,7,3590.44,15,25133.08,21363.12,2025-03-13,2025-03-14,2025-03-16,Kerala,682092,2026-09-16T11:51:02.552Z,orders_day2.csv,BATCH002,ecommerce_app,1,PAID,Delivered,DEBIT CARD,FASHION,Kochi,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100244,CUST10112,Vinod Chowdary,vinod.chowdary@hotmail.com,6248578017,PROD1082,Hair Serum,8,1043.93,0,8351.44,8351.44,2025-10-15,null,null,Telangana,500048,2026-09-16T11:51:02.552Z,orders_day2.csv,BATCH002,ecommerce_app,1,FAILED,Cancelled,WALLET,BEAUTY,Hyderabad,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100428,CUST10037,Kalpana Rajan,kalpana.rajan@outlook.com,9655742829,PROD1089,Wireless Headphones,1,3511.57,40,3511.57,2106.94,2025-01-23,2025-01-24,2025-01-27,Rajasthan,302079,2026-09-16T11:51:02.552Z,orders_day2.csv,BATCH002,ecommerce_app,1,PAID,Delivered,CASH ON DELIVERY,ELECTRONICS,Jaipur,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100111,CUST10130,Rakesh Murthy,rakesh.murthy@outlook.com,7750572325,PROD1059,Herbal Face Wash,4,287.81,10,1151.24,1036.12,2025-01-26,2025-01-30,2025-02-02,Karnataka,560055,2026-09-16T11:51:02.552Z,orders_day2.csv,BATCH002,ecommerce_app,1,PAID,Delivered,UPI,BEAUTY,Bengaluru,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100958,CUST10098,Selvam Varma,selvam.varma@yahoo.com,9952266893,PROD1071,LED Monitor,5,14302.16,15,71510.8,60784.18,2026-01-06,null,null,Delhi,110014,2026-09-16T11:51:02.552Z,orders_day2.csv,BATCH002,ecommerce_app,1,PENDING,Placed,CREDIT CARD,ELECTRONICS,Delhi,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100959,CUST10050,Ashwin Menon,ashwin.menon@gmail.com,6200140141,PROD1100,Mixer Grinder,2,10581.53,0,21163.06,21163.06,2026-01-05,null,null,Kerala,682040,2026-09-16T11:51:02.552Z,orders_day2.csv,BATCH002,ecommerce_app,1,PENDING,Placed,WALLET,HOME,Kochi,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,,true
ORD100960,CUST99994,Priya Siddiqui,priya.siddiqui@yahoo.com,6359154084,PROD1082,Hair Serum,1,321.12,5,321.12,305.06,2026-01